In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

In [3]:
def uji_statistik_aes(preds_baseline, preds_proposed, trait_name):
    print(f"==================================================")
    print(f"UJI STATISTIK UNTUK TRAIT: {trait_name.upper()}")
    print(f"==================================================")
    
    # 1. Ekstraksi dan Perhitungan Selisih Skor
    diff = preds_proposed - preds_baseline
    
    # 2. Evaluasi Distribusi dan Analisis Momen
    skewness = stats.skew(diff)
    kurtosis = stats.kurtosis(diff)
    
    print(f"1. Analisis Momen Distribusi Selisih:")
    print(f"   - Skewness : {skewness:.4f}")
    print(f"   - Kurtosis : {kurtosis:.4f}")
    
    # Menentukan threshold anomali ekstrem
    # (Nilai umum untuk sampel besar: skewness > |2|, kurtosis > |7|)
    is_extreme = abs(skewness) > 2.0 or abs(kurtosis) > 7.0
    
    # 3. Pemilihan dan Eksekusi Uji Hipotesis Berpasangan
    print(f"\n2. Pemilihan Uji Hipotesis:")
    if not is_extreme:
        print("   -> Distribusi TIDAK ekstrem. Menggunakan Paired Sample T-Test (Metode Utama).")
        stat, p_value = stats.ttest_rel(preds_proposed, preds_baseline)
    else:
        print("   -> Ditemukan anomali ekstrem. Menggunakan Wilcoxon Signed-Rank Test (Metode Alternatif).")
        stat, p_value = stats.wilcoxon(preds_proposed, preds_baseline)
        
    print(f"   - Statistik Uji : {stat:.4f}")
    print(f"   - p-value       : {p_value:.4e}")
    
    # 4. Penarikan Kesimpulan Hipotesis
    print(f"\n3. Penarikan Kesimpulan:")
    if p_value < 0.05:
        print("   -> KESIMPULAN: Tolak H0 (p < 0.05).")
        print("      Terdapat perbedaan performa prediksi yang SIGNIFIKAN secara statistik")
        print("      antara model baseline (CLIP) dan model usulan (RoBERTa).")
    else:
        print("   -> KESIMPULAN: Gagal Tolak H0 (p >= 0.05).")
        print("      TIDAK terdapat perbedaan performa prediksi yang signifikan secara")
        print("      statistik antara kedua model.")
    print("\n")

In [ ]:
# Load dataset
df_clip = pd.read_csv('clip_predictions_slidingwindow.csv')
df_roberta_clarity = pd.read_csv('multimodal/roberta_test_clarity.csv')

preds_clip_clarity = df_clip['argument_clarity(prediction)'].values
preds_roberta_clarity = df_roberta_clarity['prediction'].values

min_len_clarity = min(len(preds_clip_clarity), len(preds_roberta_clarity))
preds_clip_clarity = preds_clip_clarity[:min_len_clarity]
preds_roberta_clarity = preds_roberta_clarity[:min_len_clarity]

# Eksekusi fungsi
uji_statistik_aes(preds_clip_clarity, preds_roberta_clarity, "Argument Clarity")

UJI STATISTIK UNTUK TRAIT: ARGUMENT CLARITY
1. Analisis Momen Distribusi Selisih:
   - Skewness : 0.2332
   - Kurtosis : -0.0939

2. Pemilihan Uji Hipotesis:
   -> Distribusi TIDAK ekstrem. Menggunakan Paired Sample T-Test (Metode Utama).
   - Statistik Uji : 1.0270
   - p-value       : 3.0576e-01

3. Penarikan Kesimpulan:
   -> KESIMPULAN: Gagal Tolak H0 (p >= 0.05).
      TIDAK terdapat perbedaan performa prediksi yang signifikan secara
      statistik antara kedua model.




In [5]:
# Load dataset
df_roberta_persuasive = pd.read_csv('multimodal/roberta_test_persuasive.csv')

# Ekstraksi array prediksi untuk Justifying Persuasiveness
preds_clip_persuasive = df_clip['justifying_persuasiveness(prediction)'].values
preds_roberta_persuasive = df_roberta_persuasive['prediction'].values

# Menyamakan jumlah sampel
min_len_persuasive = min(len(preds_clip_persuasive), len(preds_roberta_persuasive))
preds_clip_persuasive = preds_clip_persuasive[:min_len_persuasive]
preds_roberta_persuasive = preds_roberta_persuasive[:min_len_persuasive]

# Eksekusi fungsi
uji_statistik_aes(preds_clip_persuasive, preds_roberta_persuasive, "Justifying Persuasiveness")

UJI STATISTIK UNTUK TRAIT: JUSTIFYING PERSUASIVENESS
1. Analisis Momen Distribusi Selisih:
   - Skewness : -0.0764
   - Kurtosis : -0.4143

2. Pemilihan Uji Hipotesis:
   -> Distribusi TIDAK ekstrem. Menggunakan Paired Sample T-Test (Metode Utama).
   - Statistik Uji : 10.5182
   - p-value       : 1.3348e-20

3. Penarikan Kesimpulan:
   -> KESIMPULAN: Tolak H0 (p < 0.05).
      Terdapat perbedaan performa prediksi yang SIGNIFIKAN secara statistik
      antara model baseline (CLIP) dan model usulan (RoBERTa).




In [ ]:
df_roberta_text = pd.read_csv(r'Kriteria Tekstual\roberta_test.csv')

for col_name in df_roberta_text.columns:
    clip_col_name = f"{col_name}(prediction)"
    
    if clip_col_name in df_clip.columns:
        
        # Ekstraksi array prediksi
        preds_clip_text = df_clip[clip_col_name].values
        preds_roberta_text = df_roberta_text[col_name].values
        
        # Menyamakan jumlah sampel (mencegah error jika jumlah baris berbeda)
        min_len_text = min(len(preds_clip_text), len(preds_roberta_text))
        preds_clip_text = preds_clip_text[:min_len_text]
        preds_roberta_text = preds_roberta_text[:min_len_text]
        
        # Memformat nama trait untuk dicetak (contoh: "Organizational Structure")
        trait_display_name = str(col_name).replace('_', ' ').title()
        
        # Eksekusi fungsi uji statistik
        uji_statistik_aes(preds_clip_text, preds_roberta_text, trait_display_name)
        
    else:
        pass

UJI STATISTIK UNTUK TRAIT: ORGANIZATIONAL STRUCTURE
1. Analisis Momen Distribusi Selisih:
   - Skewness : -0.0302
   - Kurtosis : -0.3614

2. Pemilihan Uji Hipotesis:
   -> Distribusi TIDAK ekstrem. Menggunakan Paired Sample T-Test (Metode Utama).
   - Statistik Uji : 10.5486
   - p-value       : 1.0898e-20

3. Penarikan Kesimpulan:
   -> KESIMPULAN: Tolak H0 (p < 0.05).
      Terdapat perbedaan performa prediksi yang SIGNIFIKAN secara statistik
      antara model baseline (CLIP) dan model usulan (RoBERTa).


UJI STATISTIK UNTUK TRAIT: COHERENCE
1. Analisis Momen Distribusi Selisih:
   - Skewness : -0.1271
   - Kurtosis : -0.3412

2. Pemilihan Uji Hipotesis:
   -> Distribusi TIDAK ekstrem. Menggunakan Paired Sample T-Test (Metode Utama).
   - Statistik Uji : 15.9920
   - p-value       : 8.8601e-37

3. Penarikan Kesimpulan:
   -> KESIMPULAN: Tolak H0 (p < 0.05).
      Terdapat perbedaan performa prediksi yang SIGNIFIKAN secara statistik
      antara model baseline (CLIP) dan model usula